Construção da camada Silver

In [1]:
import os
import platform
from datetime import datetime
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, trim, when

In [2]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('construcao_camada_silver')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('Versão do Spark:', spark.version)

Versão do Spark: 3.5.9


Localização das pastas

Primeiro identificamos a raiz do projeto.

In [3]:
pasta_atual = Path.cwd()

if pasta_atual.name == 'notebooks':
    raiz_projeto = pasta_atual.parent
else:
    raiz_projeto = pasta_atual

pasta_bronze = raiz_projeto / 'data' / 'bronze'
pasta_silver = raiz_projeto / 'data' / 'silver'
pasta_silver.mkdir(parents=True, exist_ok=True)

print('Bronze:', pasta_bronze)
print('Silver:', pasta_silver)

Bronze: C:\Users\claud\Documents\Py\tech_challenge_02\data\bronze
Silver: C:\Users\claud\Documents\Py\tech_challenge_02\data\silver


Regras das tabelas

Para cada tabela informamos a chave usada para procurar duplicidades. Também separamos as colunas inteiras e decimais. Os identificadores continuam como texto para não perder zeros à esquerda.

In [4]:
chaves_tabelas = {
    'alunos': ['ano', 'id_aluno'],
    'meta_alfabetizacao_brasil': ['ano', 'rede'],
    'meta_alfabetizacao_municipio': ['ano', 'id_municipio', 'rede'],
    'meta_alfabetizacao_uf': ['ano', 'sigla_uf', 'rede'],
    'municipio': ['ano', 'id_municipio', 'serie', 'rede'],
    'uf': ['ano', 'sigla_uf', 'serie', 'rede'],
}

colunas_inteiras = [
    'ano', 'caderno', 'serie', 'presenca',
    'preenchimento_caderno', 'alfabetizado', 'nivel_alfabetizacao'
]

colunas_decimais = [
    'proficiencia', 'peso_aluno', 'taxa_alfabetizacao',
    'media_portugues', 'percentual_participacao',
    'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025',
    'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027',
    'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029',
    'meta_alfabetizacao_2030',
    'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1',
    'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3',
    'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5',
    'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7',
    'proporcao_aluno_nivel_8'
]

Verificação do ambiente

O Hadoop utilizado pelo Spark precisa do programa `winutils.exe` no Windows. Quando ele não estiver instalado, usamos o caminho alternativo para teste local.

In [5]:
hadoop_home = os.getenv('HADOOP_HOME')
winutils_existe = bool(
    hadoop_home and (Path(hadoop_home) / 'bin' / 'winutils.exe').exists()
)
usar_spark = platform.system() != 'Windows' or winutils_existe


Tratamento com PySpark

A função abaixo limpa os espaços dos textos, corrige o nome, converte tipos e remove duplicidades usando a chave de cada tabela.

In [6]:
def tratar_com_spark(nome_tabela, caminho_bronze):
    df = spark.read.parquet(caminho_bronze.as_posix())

    for nome_coluna, tipo_coluna in df.dtypes:
        if tipo_coluna == 'string':
            df = df.withColumn(nome_coluna, trim(col(nome_coluna)))

    if 'rede' in df.columns:
        df = df.withColumn(
            'rede',
            when(col('rede') == 'P�blica', 'Pública').otherwise(col('rede'))
        )

    for nome_coluna in colunas_inteiras:
        if nome_coluna in df.columns:
            df = df.withColumn(nome_coluna, col(nome_coluna).cast('integer'))

    for nome_coluna in colunas_decimais:
        if nome_coluna in df.columns:
            df = df.withColumn(nome_coluna, col(nome_coluna).cast('double'))

    if '_data_ingestao' in df.columns:
        df = df.withColumn('_data_ingestao', col('_data_ingestao').cast('timestamp'))

    df = df.dropDuplicates(chaves_tabelas[nome_tabela])
    df = df.withColumn('_data_tratamento', current_timestamp())
    return df

Tratamento local no Windows

Esta função repete as regras da Silver com Pandas. Ela existe somente para conseguirmos executar o protótipo no Windows sem configurar o Hadoop.

In [7]:
def tratar_com_pandas(nome_tabela, arquivo_bronze):
    df = pd.read_parquet(arquivo_bronze)

    for nome_coluna in df.select_dtypes(include=['object', 'string']).columns:
        df[nome_coluna] = df[nome_coluna].astype('string').str.strip()

    if 'rede' in df.columns:
        df['rede'] = df['rede'].replace({'P�blica': 'Pública'})

    for nome_coluna in colunas_inteiras:
        if nome_coluna in df.columns:
            df[nome_coluna] = pd.to_numeric(df[nome_coluna], errors='coerce').astype('Int64')

    for nome_coluna in colunas_decimais:
        if nome_coluna in df.columns:
            df[nome_coluna] = pd.to_numeric(df[nome_coluna], errors='coerce')

    if '_data_ingestao' in df.columns:
        df['_data_ingestao'] = pd.to_datetime(df['_data_ingestao'], errors='coerce')

    df = df.drop_duplicates(subset=chaves_tabelas[nome_tabela])
    df['_data_tratamento'] = pd.Timestamp.now()
    return df

Construção da camada Silver


In [8]:
dados_silver = {}
caminhos_silver = {}

for nome_tabela in chaves_tabelas:
    pasta_tabela_bronze = pasta_bronze / nome_tabela
    cargas = sorted(pasta_tabela_bronze.glob('id_ingestao=*'))

    if not cargas:
        raise FileNotFoundError(f'Não existe carga Bronze para {nome_tabela}')

    carga_mais_recente = cargas[-1]
    caminho_saida = pasta_silver / nome_tabela

    if usar_spark:
        df_silver = tratar_com_spark(nome_tabela, carga_mais_recente)
        df_silver.write.mode('overwrite').parquet(caminho_saida.as_posix())
    else:
        arquivos_parquet = sorted(carga_mais_recente.glob('*.parquet'))
        if not arquivos_parquet:
            raise FileNotFoundError(f'Arquivo Parquet não encontrado para {nome_tabela}')
        df_silver = tratar_com_pandas(nome_tabela, arquivos_parquet[0])
        caminho_saida.mkdir(parents=True, exist_ok=True)
        df_silver.to_parquet(caminho_saida / 'data.parquet', index=False)

    dados_silver[nome_tabela] = df_silver
    caminhos_silver[nome_tabela] = caminho_saida
    print('Tabela Silver gravada:', nome_tabela)

Tabela Silver gravada: alunos


Tabela Silver gravada: meta_alfabetizacao_brasil


Tabela Silver gravada: meta_alfabetizacao_municipio


Tabela Silver gravada: meta_alfabetizacao_uf


Tabela Silver gravada: municipio


Tabela Silver gravada: uf


Validação

In [9]:
resultado_validacao = []

for nome_tabela, df in dados_silver.items():
    chaves = chaves_tabelas[nome_tabela]

    if usar_spark:
        total = df.count()
        total_sem_duplicidade = df.dropDuplicates(chaves).count()
        filtro_nulos = None
        for chave in chaves:
            condicao = col(chave).isNull()
            filtro_nulos = condicao if filtro_nulos is None else filtro_nulos | condicao
        chaves_nulas = df.filter(filtro_nulos).count()
    else:
        total = len(df)
        total_sem_duplicidade = len(df.drop_duplicates(subset=chaves))
        chaves_nulas = int(df[chaves].isna().any(axis=1).sum())

    resultado_validacao.append({
        'tabela': nome_tabela,
        'registros': total,
        'duplicidades': total - total_sem_duplicidade,
        'chaves_nulas': chaves_nulas,
    })

pd.DataFrame(resultado_validacao)

,tabela,registros,duplicidades,chaves_nulas
0,alunos,3867999,0,0
1,meta_alfabetizacao_brasil,3,0,0
2,meta_alfabetizacao_municipio,10704,0,0
3,meta_alfabetizacao_uf,54,0,0
4,municipio,23995,0,0
5,uf,145,0,0


Visualização da tabela de alunos

In [10]:
if usar_spark:
    dados_silver['alunos'].printSchema()
    dados_silver['alunos'].show(5, truncate=False)
else:
    display(dados_silver['alunos'].head())
    print(dados_silver['alunos'].dtypes)

In [11]:
spark.stop()